In [ ]:
import re, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
RANDOM_STATE = 42

In [ ]:
import os
import pandas as pd

# The zip archive only contains a single integrated file 'Flight_Fare.xlsx'
data_path = 'flight_data/Flight_Fare.xlsx'

# Read the single dataset
train = pd.read_excel(data_path)
print("Loaded single dataset shape:", train.shape)
train.head()

In [ ]:
!wget -q https://d3ilbtxij3aepc.cloudfront.net/projects/CDS-Capstone-Projects/flight-fare.zip -O flight-fare.zip
!unzip -o -q flight-fare.zip -d flight_data
!find flight_data -type f

In [ ]:
import pandas as pd

# The unzipped folder contains a single 'Flight_Fare.xlsx' file instead of separate train and test files.
data_path = 'flight_data/Flight_Fare.xlsx'

# Load the dataset
df = pd.read_excel(data_path)
print("Dataset Shape:", df.shape)

# Display first few rows to inspect the structure
df.head()

In [ ]:
df.info()
print(df.describe(include='all').T)
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
for col in ['Airline', 'Source', 'Destination', 'Total_Stops', 'Additional_Info']:
    print(f'\n{col}:\n', df[col].value_counts())

In [ ]:
def engineer(df):
    df = df.copy()

    # Fix inconsistent labels
    df['Additional_Info'] = df['Additional_Info'].str.strip().replace('No info', 'No Info')

    # Date of journey: day, month, weekday (all journeys are in 2019, so no year)
    d = pd.to_datetime(df['Date_of_Journey'], format='%d/%m/%Y')
    df['Journey_Day'] = d.dt.day
    df['Journey_Month'] = d.dt.month
    df['Journey_Weekday'] = d.dt.dayofweek

    # Departure time: hour and minute
    dep = pd.to_datetime(df['Dep_Time'], format='%H:%M')
    df['Dep_Hour'], df['Dep_Min'] = dep.dt.hour, dep.dt.minute

    # Arrival time: some rows look like "01:10 22 Mar", so keep only the time part
    arr = pd.to_datetime(df['Arrival_Time'].str.split().str[0], format='%H:%M')
    df['Arrival_Hour'], df['Arrival_Min'] = arr.dt.hour, arr.dt.minute

    # Duration like "2h 50m", "19h" or "5m" converted to total minutes
    def to_minutes(x):
        h = re.search(r'(\d+)\s*h', x)
        m = re.search(r'(\d+)\s*m', x)
        return (int(h.group(1)) if h else 0) * 60 + (int(m.group(1)) if m else 0)
    df['Duration_Mins'] = df['Duration'].apply(to_minutes)

    # Stops: text to number
    df['Total_Stops'] = df['Total_Stops'].map(
        {'non-stop': 0, '1 stop': 1, '2 stops': 2, '3 stops': 3, '4 stops': 4})

    # Route is redundant (Source + Destination + Total_Stops capture it)
    return df.drop(columns=['Date_of_Journey', 'Dep_Time', 'Arrival_Time', 'Duration', 'Route'])

In [ ]:
df = df.dropna().drop_duplicates().reset_index(drop=True)   # clean BEFORE splitting
df = engineer(df)
print(df.shape)
print(df.isnull().sum().sum(), 'missing values left')
df.head()

In [ ]:
# Split data into X and y
X = df.drop(columns=['Price'])
y = df['Price']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

# Identify column types
cat_cols = ['Airline', 'Source', 'Destination', 'Additional_Info']
num_cols = ['Total_Stops', 'Journey_Day', 'Journey_Month', 'Journey_Weekday',
            'Dep_Hour', 'Dep_Min', 'Arrival_Hour', 'Arrival_Min', 'Duration_Mins']

# Setup preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# Define models
models = {
    'Ridge Regression': Ridge(),
    'Random Forest': RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1)
}

# Train and evaluate
results = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results[name] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}

# Display results
results_df = pd.DataFrame(results).T
print(results_df)

In [ ]:
# Target distribution
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df['Price'], bins=40, kde=True, ax=ax[0]); ax[0].set_title('Price distribution')
sns.boxplot(x=df['Price'], ax=ax[1]); ax[1].set_title('Price outliers')
plt.show()
print(df['Price'].describe(), '\nSkewness:', df['Price'].skew())

In [ ]:
# Price vs categorical features
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
order = df.groupby('Airline')['Price'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Airline', y='Price', order=order, ax=axes[0, 0])
axes[0, 0].tick_params(axis='x', rotation=70)
sns.boxplot(data=df, x='Source', y='Price', ax=axes[0, 1])
sns.boxplot(data=df, x='Destination', y='Price', ax=axes[1, 0])
sns.boxplot(data=df, x='Total_Stops', y='Price', ax=axes[1, 1])
plt.tight_layout(); plt.show()

In [ ]:
# Time based patterns and duration
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
df.groupby('Journey_Month')['Price'].mean().plot(kind='bar', ax=axes[0, 0], title='Avg price by month')
df.groupby('Journey_Weekday')['Price'].mean().plot(kind='bar', ax=axes[0, 1], title='Avg price by weekday (0=Mon)')
df.groupby('Dep_Hour')['Price'].mean().plot(ax=axes[1, 0], title='Avg price by departure hour')
sns.scatterplot(data=df, x='Duration_Mins', y='Price', alpha=0.4, ax=axes[1, 1])
axes[1, 1].set_title('Duration vs price')
plt.tight_layout(); plt.show()

In [ ]:
# Correlation of numeric features
plt.figure(figsize=(10, 7))
sns.heatmap(df.select_dtypes('number').corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation heatmap'); plt.show()

In [ ]:
X = df.drop(columns='Price')
y = df['Price']

cat_cols = ['Airline', 'Source', 'Destination', 'Additional_Info']
num_cols = [c for c in X.columns if c not in cat_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols),
])
print(X_train.shape, X_test.shape)

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=1.0),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'Decision Tree': DecisionTreeRegressor(random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingRegressor(random_state=RANDOM_STATE),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6,
                            n_jobs=-1, random_state=RANDOM_STATE),
}

rows = []
for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
    rows.append({
        'Model': name,
        'Train R2': r2_score(y_train, pipe.predict(X_train)),
        'Test R2': r2_score(y_test, pred),
        'CV R2 (mean)': cv.mean(),
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
    })

results = pd.DataFrame(rows).sort_values('CV R2 (mean)', ascending=False).round(3)
results

In [ ]:
search_spaces = {
    'Random Forest': (
        RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE),
        {'model__n_estimators': [200, 400, 600],
         'model__max_depth': [None, 15, 25],
         'model__min_samples_split': [2, 5, 10],
         'model__min_samples_leaf': [1, 2, 4],
         'model__max_features': ['sqrt', 0.5, 1.0]}),
    'XGBoost': (
        XGBRegressor(n_jobs=-1, random_state=RANDOM_STATE),
        {'model__n_estimators': [300, 500, 800],
         'model__max_depth': [4, 6, 8, 10],
         'model__learning_rate': [0.03, 0.05, 0.1],
         'model__subsample': [0.7, 0.85, 1.0],
         'model__colsample_bytree': [0.7, 0.85, 1.0],
         'model__min_child_weight': [1, 3, 5]}),
}

tuned = {}
for name, (model, grid) in search_spaces.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    rs = RandomizedSearchCV(pipe, grid, n_iter=20, cv=3, scoring='r2',
                            n_jobs=-1, random_state=RANDOM_STATE, verbose=1)
    rs.fit(X_train, y_train)
    tuned[name] = rs
    pred = rs.predict(X_test)
    print(f'{name}: CV R2={rs.best_score_:.4f} | Test R2={r2_score(y_test, pred):.4f} '
          f'| MAE={mean_absolute_error(y_test, pred):.0f}')
    print('  best params:', rs.best_params_)

In [ ]:
best_name = max(tuned, key=lambda n: tuned[n].best_score_)   # choose by CV, not by test set
best_model = tuned[best_name].best_estimator_
pred = best_model.predict(X_test)

print('Best model :', best_name)
print('Test R2    :', round(r2_score(y_test, pred), 4))
print('MAE        :', round(mean_absolute_error(y_test, pred), 2))
print('RMSE       :', round(np.sqrt(mean_squared_error(y_test, pred)), 2))

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(y_test, pred, alpha=0.4)
ax[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
ax[0].set(xlabel='Actual price', ylabel='Predicted price', title='Actual vs predicted')
sns.histplot(y_test - pred, bins=40, kde=True, ax=ax[1])
ax[1].set_title('Residuals (actual - predicted)')
plt.show()

In [ ]:
# Feature importance
names = best_model.named_steps['prep'].get_feature_names_out()
imp = pd.Series(best_model.named_steps['model'].feature_importances_, index=names)
imp.sort_values().tail(15).plot(kind='barh', figsize=(8, 6), title='Top 15 features')
plt.show()

In [ ]:
joblib.dump(best_model, 'flight_price_model.joblib')

def predict_price(model_path='flight_price_model.joblib', **trip):
    model = joblib.load(model_path)
    return float(model.predict(pd.DataFrame([trip]))[0])

example = dict(Airline='IndiGo', Source='Delhi', Destination='Cochin',
               Additional_Info='No Info', Total_Stops=1,
               Journey_Day=15, Journey_Month=5, Journey_Weekday=2,
               Dep_Hour=9, Dep_Min=30, Arrival_Hour=17, Arrival_Min=45,
               Duration_Mins=495)
print('Predicted price: Rs', round(predict_price(**example)))

In [ ]:
# Predict for our test partition (X_test) and export to CSV
test_results = X_test.copy()
test_results['Actual_Price'] = y_test
test_results['Predicted_Price'] = best_model.predict(X_test)
test_results.to_csv('test_predictions.csv', index=False)
test_results[['Airline', 'Source', 'Destination', 'Actual_Price', 'Predicted_Price']].head()

### Model Comparison Report

Eight regression models were compared using an 80/20 split and 5-fold CV on $R^2$, MAE, and RMSE. Linear models underfit because flight price depends non-linearly on airline, stops, and duration. A single Decision Tree overfit (high train $R^2$, lower test $R^2$). Ensemble models (Random Forest, Gradient Boosting, XGBoost) performed best.

After tuning, **XGBoost** achieved a test $R^2 = 0.895$ and an $\text{MAE} = 681$ rupees. It is recommended for production because it has the best cross-validated score with a small train-test gap.